# ACT を学習する（Colab）

oracle のデモ (`episode_XXXX.npz`) を LeRobot のデータセットに変換し、lerobot 公式の ACT を学習する.
チェックポイントは Google Drive に保存するので、途中で切れても上から実行し直せば続きから再開する.

**準備（1回だけ）**: Google Drive に次のように置く（フォルダ名は下の設定セルで変えられる）.

```
MyDrive/pai_sim/
  tools/to_lerobot.py, tools/inspect_demo.py   ← このリポジトリの tools/ から
  demos/oracle_n1_s2000/episode_0000.npz ...   ← pai_eval_data/oracle_n1_s2000/demos/ の中身
```

**ランタイム**: 「ランタイムのタイプを変更」で T4 GPU を選ぶ.

| 出力（Drive） | 中身 |
|---|---|
| `datasets/<DATASET_NAME>/` | 変換したデータセット（2回目からはこれを使い、変換を飛ばす） |
| `runs/<RUN_NAME>/checkpoint/` | 最新のチェックポイント（`pretrained_model/` が推論に使うもの） |
| `runs/<RUN_NAME>/train.log`, `offline_eval.png` | 学習のログ、取り分けたエピソードでの予測 |

lerobot 0.6.1 で手元 (CPU) で数ステップ動かして確認したコマンドを使っている. 版を変えるとデータセット形式や引数が変わるので注意.

In [ ]:
# ---- 設定 ----
DRIVE_DIR = '/content/drive/MyDrive/pai_sim'
DEMO_DIRS = ['demos/oracle_n1_s2000', 'demos/oracle_n1_s2100']   # DRIVE_DIR からの相対パス. 複数並べるとまとめて1つのデータセットにする
DATASET_NAME = 'oracle_n1_s2000_2100_n300_trim'   # 最初の止まっているフレームを落としたもの
RUN_NAME = 'act_n1_s2000_2100_n300_trim'   # 落とす前の act_n1_s2000_2100_n300（4 万ステップ）は Gazebo で 2/10
VAL_EPISODES = 10      # 最後の何エピソードを学習に使わず、予測の確認用に取っておくか
STEPS = 40_000         # デモを 3 倍にしたので増やす（300 本で約 5.5 エポック）. 途中で切れても CHUNK ごとに Drive から再開できる
CHUNK = 2_500          # この刻みで学習を区切り、毎回 Drive にチェックポイントを保存する
BATCH_SIZE = 8         # ACT の既定
NUM_WORKERS = 2        # Colab の CPU は 2 コア
TRIM_IDLE = 0.05      # 最初にアームが止まっているフレームを落とすしきい値 [rad]. 0 なら落とさない
LEROBOT_VERSION = '0.6.1'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
assert os.path.exists(f'{DRIVE_DIR}/tools/to_lerobot.py'), f'{DRIVE_DIR}/tools/ に to_lerobot.py と inspect_demo.py を置く'
for d in DEMO_DIRS:
    n = len([f for f in os.listdir(f'{DRIVE_DIR}/{d}') if f.endswith('.npz')])
    print(f'{d}: {n} demos')

In [ ]:
%pip install -q "lerobot[dataset,training]=={LEROBOT_VERSION}"
# Colab に最初から入っている torchao は diffusers と版が合わず、lerobot-train が import で落ちる（ACT では使わない）
%pip uninstall -y -q torchao

`%pip install` のあと「セッションを再起動」と出たら、再起動して **設定セルから** 実行し直す（install は飛ばしてよい）.

In [ ]:
import importlib.metadata as md
import torch
print({p: md.version(p) for p in ['lerobot', 'torch', 'torchvision', 'torchcodec', 'av']})
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'GPU のランタイムになっていない'

## 1. データセットに変換する

Drive に変換済みのものがあればそれをコピーし、なければデモを変換して Drive にも保存する.

In [ ]:
import glob, shutil, subprocess

DS_LOCAL = f'/content/datasets/{DATASET_NAME}'
DS_DRIVE = f'{DRIVE_DIR}/datasets/{DATASET_NAME}'

if os.path.exists(f'{DS_LOCAL}/meta/info.json'):
    print('local dataset exists:', DS_LOCAL)
elif os.path.exists(f'{DS_DRIVE}/meta/info.json'):
    shutil.copytree(DS_DRIVE, DS_LOCAL)
    print('copied from Drive:', DS_DRIVE)
else:
    # Drive から直接読むと遅いので、先にローカルへコピーしてから変換する
    shutil.copytree(f'{DRIVE_DIR}/tools', '/content/tools', dirs_exist_ok=True)
    local_demos = []
    for d in DEMO_DIRS:
        dst = f'/content/demos/{d}'
        shutil.copytree(f'{DRIVE_DIR}/{d}', dst, dirs_exist_ok=True)
        local_demos.append(dst)
    subprocess.run(['python', '/content/tools/to_lerobot.py', *local_demos, '--root', DS_LOCAL,
                    '--repo-id', f'local/{DATASET_NAME}', '--trim-idle', str(TRIM_IDLE), '--overwrite'], check=True)
    shutil.copytree(DS_LOCAL, DS_DRIVE, dirs_exist_ok=True)
    print('saved to Drive:', DS_DRIVE)
print(open(f'{DS_LOCAL}/source.json').read())

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

# 動画の読み込みは torchcodec が速い. torch と版が合わずに読めないときは pyav にする
VIDEO_BACKEND = 'pyav'
try:
    LeRobotDataset(f'local/{DATASET_NAME}', root=DS_LOCAL, video_backend='torchcodec')[0]
    VIDEO_BACKEND = 'torchcodec'
except Exception as e:
    print('torchcodec unavailable:', repr(e)[:200])
print('video backend:', VIDEO_BACKEND)

ds = LeRobotDataset(f'local/{DATASET_NAME}', root=DS_LOCAL, video_backend=VIDEO_BACKEND)
N_EP = ds.num_episodes
TRAIN_EPISODES = list(range(N_EP - VAL_EPISODES))
VAL_EPISODE_IDS = list(range(N_EP - VAL_EPISODES, N_EP))
print(f'episodes: {N_EP} (train {len(TRAIN_EPISODES)}, val {len(VAL_EPISODE_IDS)}), frames: {ds.num_frames}, fps: {ds.fps}')
print('epochs at STEPS:', round(STEPS * BATCH_SIZE / ds.num_frames, 1))

## 2. 学習する

`CHUNK` ステップごとに `lerobot-train` を区切って実行し、そのたびに最新のチェックポイントを Drive に保存する
（ACT は学習率が一定なので、区切っても1回で通したのと同じ）. Drive に前回のチェックポイントがあれば、そこから再開する.

In [ ]:
import json, pathlib

OUT = pathlib.Path(f'/content/outputs/{RUN_NAME}')
RUN_DRIVE = pathlib.Path(f'{DRIVE_DIR}/runs/{RUN_NAME}')
LOG = RUN_DRIVE / 'train.log'
RUN_DRIVE.mkdir(parents=True, exist_ok=True)


def step_of(ckpt):
    return json.loads((ckpt / 'training_state' / 'training_step.json').read_text())['step']


def local_last():
    last = OUT / 'checkpoints' / 'last'
    return last.resolve() if last.exists() else None


def restore():
    """Drive のチェックポイントをローカルの元の場所に戻す（train_config.json がその場所を指している）."""
    src = RUN_DRIVE / 'checkpoint'
    if local_last() or not src.exists():
        return
    dst = OUT / 'checkpoints' / src.joinpath('name.txt').read_text().strip()
    shutil.copytree(src, dst)
    (OUT / 'checkpoints' / 'last').symlink_to(dst.name)
    print('restored', dst)


def backup():
    """最新のチェックポイントを Drive に上書き保存し、ローカルの古いものを消す."""
    ckpt = local_last()
    tmp, dst = RUN_DRIVE / 'checkpoint_tmp', RUN_DRIVE / 'checkpoint'
    shutil.rmtree(tmp, ignore_errors=True)
    shutil.copytree(ckpt, tmp)
    (tmp / 'name.txt').write_text(ckpt.name)
    shutil.rmtree(dst, ignore_errors=True)
    tmp.rename(dst)
    for old in (OUT / 'checkpoints').iterdir():
        if old.name not in ('last', ckpt.name):
            shutil.rmtree(old)
    print(f'backed up step {step_of(ckpt)} -> {dst}')


def run(args):
    """lerobot-train を動かし、ログは全部 Drive に、学習の進み具合の行だけ画面に出す."""
    with open(LOG, 'a', encoding='utf-8') as log:
        p = subprocess.Popen(['lerobot-train', *args], stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1,
                             encoding='utf-8', errors='replace')
        for line in p.stdout:
            log.write(line)
            if ' step:' in line or 'Error' in line or 'Traceback' in line:
                print(line.rstrip()[-160:], flush=True)
    if p.wait() != 0:
        raise RuntimeError(f'lerobot-train failed (exit {p.returncode}). see {LOG}')


restore()

In [ ]:
import time

while True:
    last = local_last()
    done = step_of(last) if last else 0
    if done >= STEPS:
        print('finished:', done, 'steps')
        break
    target = min(done + CHUNK, STEPS)
    t0 = time.time()
    if last is None:
        shutil.rmtree(OUT, ignore_errors=True)   # 前回の失敗で残った出力があると lerobot-train が止まる
        run([f'--dataset.repo_id=local/{DATASET_NAME}', f'--dataset.root={DS_LOCAL}',
             f'--dataset.episodes={TRAIN_EPISODES}', f'--dataset.video_backend={VIDEO_BACKEND}',
             '--policy.type=act', '--policy.device=cuda', '--policy.push_to_hub=false',
             f'--output_dir={OUT}', f'--job_name={RUN_NAME}', f'--batch_size={BATCH_SIZE}',
             f'--num_workers={NUM_WORKERS}', f'--steps={target}', f'--save_freq={CHUNK}',
             '--log_freq=100', '--wandb.enable=false'])
    else:
        run([f'--config_path={last}/pretrained_model/train_config.json', '--resume=true',
             f'--steps={target}', f'--save_freq={CHUNK}'])
    print(f'{done} -> {target} steps in {(time.time() - t0) / 60:.1f} min')
    backup()

In [ ]:
import re
import matplotlib.pyplot as plt


def num(s):
    """lerobot のログは 2.5K のように丸めて出す."""
    mult = {'K': 1e3, 'M': 1e6, 'B': 1e9}
    return float(s[:-1]) * mult[s[-1]] if s[-1] in mult else float(s)


rows = re.findall(r' step:([\d.]+[KMB]?) .*? loss:([\d.]+) .*? l1_loss:([\d.]+)', LOG.read_text(encoding='utf-8'))
steps, loss, l1 = (list(map(num, c)) for c in zip(*rows))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(steps, loss); ax[0].set_yscale('log'); ax[0].set_title('loss (l1 + kl_weight * kld)')
ax[1].plot(steps, l1); ax[1].set_yscale('log'); ax[1].set_title('l1_loss (normalized action)')
for a in ax:
    a.set_xlabel('step'); a.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(RUN_DRIVE / 'loss.png', dpi=100); plt.show()

## 3. 取り分けたエピソードで予測を確かめる

学習に使っていないエピソードの各フレームで、次の関節角 (`action`) を予測させて正解と比べる.
ここでずれが大きいと Gazebo でも動かないので、評価の前の目安にする（ずれが小さくても、閉ループで誤差がたまって失敗することはある）.

In [ ]:
import numpy as np
from lerobot.policies.act.modeling_act import ACTPolicy
from lerobot.policies.factory import make_pre_post_processors

CKPT = RUN_DRIVE / 'checkpoint' / 'pretrained_model'
policy = ACTPolicy.from_pretrained(str(CKPT)).to('cuda').eval()
pre, post = make_pre_post_processors(policy.config, pretrained_path=str(CKPT))
val = LeRobotDataset(f'local/{DATASET_NAME}', root=DS_LOCAL, episodes=VAL_EPISODE_IDS,
                     video_backend=VIDEO_BACKEND)
names = val.meta.features['action']['names']

pred, true, ep = [], [], []
for i in range(len(val)):
    x = val[i]
    obs = {k: x[k] for k in ('observation.images.top', 'observation.state')}
    obs['task'] = x['task']
    policy.reset()   # 毎フレーム推論し直す（チャンクの先頭だけを見る）
    with torch.no_grad():
        pred.append(post(policy.select_action(pre(obs))).cpu().numpy().ravel())
    true.append(x['action'].numpy())
    ep.append(int(x['episode_index']))
pred, true, ep = np.array(pred), np.array(true), np.array(ep)
err = np.abs(pred - true)
print('mean |error| [rad]:', dict(zip(names, err.mean(0).round(4))))
print('max  |error| [rad]:', dict(zip(names, err.max(0).round(4))))

first = ep == VAL_EPISODE_IDS[0]
fig, axes = plt.subplots(1, len(names), figsize=(18, 3))
for j, a in enumerate(axes):
    a.plot(true[first, j], label='oracle'); a.plot(pred[first, j], '--', label='ACT')
    a.set_title(names[j].replace('crane_plus_', '')); a.set_xlabel('frame')
axes[0].legend(); plt.tight_layout(); plt.savefig(RUN_DRIVE / 'offline_eval.png', dpi=100); plt.show()

## 4. 手元に持ち帰る

`MyDrive/pai_sim/runs/<RUN_NAME>/checkpoint/pretrained_model/`（約 200 MB）を Windows にダウンロードし、
Gazebo での評価（HANDOFF の手順 4）に使う. `training_state/` は学習の再開用なので持ち帰らなくてよい.